# Parse 골든셋 · PDF 문서 지표 (8건)

- **Reading order 골든**: `golden_parse.jsonl`의 `reading_order_anchors`(본문에 등장해야 할 구절을 앞→뒤 순서로 기록).
- **문서 지표**: `app.evaluation.pdf_parse_metrics` — 텍스트 레이어·레이아웃·표·제목 패턴·파서 실패율 등.
- **평가**: `pdfplumber`로 추출한 전체 텍스트에 대해 `reading_order_pass`, `structure_checks_pass` 등을 계산(파서 대리).

실행 전: **fastapi venv의 `python.exe -m pip`** 로 설치하면, CWD와 무관하게 항상 그 환경에만 들어갑니다 (`uv` 프로젝트 탐색 이슈 없음).

```text
"C:\Users\rlaeh\envs\fastapi\.venv\Scripts\python.exe" -m pip install pandas pymupdf pdfplumber
"C:\Users\rlaeh\envs\fastapi\.venv\Scripts\python.exe" -m pip install -r app/backend/requirements.txt
```

OCR 신뢰도(`ocr_confidence`)는 `pytesseract`·Tesseract 설치 시에만 채워집니다.

In [3]:
from __future__ import annotations

import json
import sys
from pathlib import Path

# 노트북 CWD가 parse 폴더이거나 루트일 때 저장소 루트 탐색
def repo_root() -> Path:
    p = Path.cwd().resolve()
    for _ in range(8):
        if (p / "data" / "golden" / "parse" / "golden_parse.jsonl").exists():
            return p
        p = p.parent
    raise FileNotFoundError("golden_parse.jsonl 을 찾을 수 없습니다. 저장소 루트에서 노트북을 여세요.")

ROOT = repo_root()
PARSE_DIR = ROOT / "data" / "golden" / "parse"
BACKEND = ROOT / "app" / "backend"
if str(BACKEND) not in sys.path:
    sys.path.insert(0, str(BACKEND))

print("ROOT =", ROOT)
print("PARSE_DIR =", PARSE_DIR)

ROOT = C:\Users\rlaeh\OneDrive\바탕 화면\Projects\RAG_finance
PARSE_DIR = C:\Users\rlaeh\OneDrive\바탕 화면\Projects\RAG_finance\data\golden\parse


## 1) 문서 지표 (8개 PDF)

In [4]:
from __future__ import annotations

import sys
from pathlib import Path


def repo_root() -> Path:
    p = Path.cwd().resolve()
    for _ in range(8):
        if (p / "data" / "golden" / "parse" / "golden_parse.jsonl").exists():
            return p
        p = p.parent
    raise FileNotFoundError(
        "golden_parse.jsonl 을 찾을 수 없습니다. 저장소 루트를 연 폴더로 열거나 터미널 CWD를 루트로 맞추세요."
    )


# 첫 셀을 건너뛰어도 `app` import 가 되도록 백엔드 경로를 여기서도 맞춤
if "ROOT" not in globals():
    ROOT = repo_root()
PARSE_DIR = ROOT / "data" / "golden" / "parse"
BACKEND = ROOT / "app" / "backend"
if str(BACKEND) not in sys.path:
    sys.path.insert(0, str(BACKEND))

import pandas as pd
from app.evaluation.pdf_parse_metrics import collect_pdf_metrics

rows = []
for pdf in sorted((PARSE_DIR / "fixtures").glob("kr_*.pdf")):
    rows.append(collect_pdf_metrics(pdf))

df_metrics = pd.DataFrame(rows)
df_metrics

ModuleNotFoundError: No module named 'numpy.rec'

ModuleNotFoundError: No module named 'numpy.rec'

## 2) 골든 reading order / 구조 검사 (pdfplumber 텍스트 기준)

동일 파이프라인을 **다른 파서**로 바꾼 뒤에도 `parsed["text"]`만 넣어 `evaluate_parsed_document`를 호출하면 됩니다.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

if "ROOT" not in globals():
    p = Path.cwd().resolve()
    for _ in range(8):
        if (p / "data" / "golden" / "parse" / "golden_parse.jsonl").exists():
            ROOT = p
            break
        p = p.parent
    else:
        raise FileNotFoundError("golden_parse.jsonl 을 찾을 수 없습니다.")

PARSE_DIR = ROOT / "data" / "golden" / "parse"
BACKEND = ROOT / "app" / "backend"
if str(BACKEND) not in sys.path:
    sys.path.insert(0, str(BACKEND))

import pandas as pd
import pdfplumber
from app.evaluation.parse_golden import evaluate_parsed_document, load_parse_golden_jsonl


def parse_pdfplumber_full(path: Path) -> tuple[str, list[dict], int]:
    parts: list[str] = []
    tables_out: list[dict] = []
    with pdfplumber.open(path) as pdf:
        n_pages = len(pdf.pages)
        for p in pdf.pages:
            parts.append(p.extract_text() or "")
            for t in p.extract_tables() or []:
                if not t:
                    continue
                headers = [str(x or "") for x in t[0]]
                rows = t[1:]
                tables_out.append({"headers": headers, "rows": rows})
    return "\n".join(parts), tables_out, n_pages


golden_path = PARSE_DIR / "golden_parse.jsonl"
items = load_parse_golden_jsonl(golden_path)
eval_rows = []
for it in items:
    fp = ROOT / it.file_path
    text, tables, n_pages = parse_pdfplumber_full(fp)
    parsed = {
        "text": text,
        "tables": tables,
        "metadata": {"parser": "pdfplumber", "total_pages": n_pages},
    }
    chunks = [c for c in text.split("\n\n") if c.strip()][:200]
    eval_rows.append(evaluate_parsed_document(it, parsed, chunks))

df_eval = pd.DataFrame(eval_rows)
df_eval

## 3) 지표 + 골든 Pass 요약보내기 (선택)

In [ ]:
out_path = PARSE_DIR / "parse_eval_snapshot.json"
snap = {
    "metrics": df_metrics.to_dict(orient="records"),
    "golden_vs_pdfplumber": df_eval.to_dict(orient="records"),
}
out_path.write_text(json.dumps(snap, ensure_ascii=False, indent=2), encoding="utf-8")
print("wrote", out_path)